# TIN-17: Layer Sensitivity Profiling with Direction Injection
### Tiny Aya — Interpretability Analysis

**Goal:** For each Tiny Aya variant (Global, Water, Earth, Fire), identify which transformer layers are most sensitive to language-specific directions injected into the residual stream.

**Pipeline overview:**
1. Load FLORES+ parallel sentences for 12 languages + English
2. Extract hidden states from each layer for each language
3. Compute a language direction per layer: `mean(lang_L) - mean(English)`
4. Sweep injection across layers — add the direction vector to the residual stream at layer `i`
5. Measure sensitivity: cosine similarity between injected and baseline representations at the final layer
6. Plot sensitivity curves and compare across model variants

> **Before running:** Make sure your Colab runtime is set to **GPU (T4)**. Runtime > Change runtime type > T4 GPU.

## 0. Setup & Installs

In [ ]:
!pip install -q transformers datasets torch accelerate huggingface_hub

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Each collaborator adds their own HF token via:
# Colab left sidebar → 🔑 Secrets → Add new secret
# Name: HF_TOKEN, Value: your token from https://huggingface.co/settings/tokens
login(token=userdata.get('HF_TOKEN'))

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from collections import defaultdict
from tqdm.auto import tqdm
import gc

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Configuration

Adjust these settings to control the scope of the analysis.

In [ ]:
# Model variants
MODEL_VARIANTS = {
    'Global': 'CohereLabs/tiny-aya-global',
    'Water':  'CohereLabs/tiny-aya-water',
    'Earth':  'CohereLabs/tiny-aya-earth',
    'Fire':   'CohereLabs/tiny-aya-fire',
}

# Target languages grouped by region — FLORES+ language codes
# English is used as the baseline direction (not in this dict)
TARGET_LANGUAGES = {
    # South Asia
    'Hindi':         'hin_Deva',
    'Bengali':       'ben_Beng',
    'Tamil':         'tam_Taml',
    # Africa
    'Swahili':       'swh_Latn',
    'Amharic':       'amh_Ethi',
    'Yoruba':        'yor_Latn',
    # West Asia
    'Arabic':        'arb_Arab',
    'Turkish':       'tur_Latn',
    'Persian':       'pes_Arab',
    # Europe
    'German':        'deu_Latn',
    'French':        'fra_Latn',
    'Spanish':       'spa_Latn',
}
ENGLISH_CODE = 'eng_Latn'

# Region mapping — used for plot colouring
LANGUAGE_REGIONS = {
    'Hindi': 'South Asia', 'Bengali': 'South Asia', 'Tamil': 'South Asia',
    'Swahili': 'Africa', 'Amharic': 'Africa', 'Yoruba': 'Africa',
    'Arabic': 'West Asia', 'Turkish': 'West Asia', 'Persian': 'West Asia',
    'German': 'Europe', 'French': 'Europe', 'Spanish': 'Europe',
}
REGION_COLORS = {
    'South Asia': '#d62728',
    'Africa':     '#2ca02c',
    'West Asia':  '#ff7f0e',
    'Europe':     '#1f77b4',
}

# Analysis settings
N_DIRECTION_SENTENCES = None   # sentences used to compute language directions
N_INJECTION_SENTENCES = 20    # sentences used for the injection sweep (batched)
INJECTION_SCALE       = 20.0  # how strongly to inject the direction vector
MAX_LENGTH            = 64    # token limit per sentence
FLORES_SPLIT          = 'dev+devtest'

## 2. Load FLORES+ Dataset

In [ ]:
def load_flores_sentences(language_code, split=FLORES_SPLIT, n=None):
    """
    Load sentences for a single language from FLORES+.
    Returns a list of strings.
    """
    dataset = load_dataset(
        'openlanguagedata/flores_plus',
        language_code,
        split='dev+devtest',
        trust_remote_code=True
    )
    sentences = [row['text'] for row in dataset]
    if n is not None:
        sentences = sentences[:n]
    return sentences

print('Loading English sentences...')
english_direction_sentences = load_flores_sentences(ENGLISH_CODE, n=N_DIRECTION_SENTENCES)

print('Loading target language sentences...')
lang_direction_sentences = {}
lang_injection_sentences = {}
for lang_name, flores_code in tqdm(TARGET_LANGUAGES.items()):
    lang_direction_sentences[lang_name] = load_flores_sentences(flores_code, n=N_DIRECTION_SENTENCES)
    lang_injection_sentences[lang_name] = load_flores_sentences(flores_code, n=N_INJECTION_SENTENCES)

print(f'Loaded sentences for {len(TARGET_LANGUAGES)} languages.')

## 3. Core Functions

### 3a. Extract Hidden States

In [ ]:
def get_mean_hidden_states(sentences, model, tokenizer, batch_size=4):
    """
    Run sentences through the model and collect mean-pooled hidden states
    at every layer (embedding layer + all transformer layers).

    Pooling is over non-padding tokens only, using the attention mask.
    Hidden states are moved to CPU immediately after each batch to avoid
    OOM on T4 (16GB VRAM is tight with a 3.35B model loaded in float16).

    Args:
        sentences:  list of strings
        model:      loaded AutoModelForCausalLM (float16, device_map='auto')
        tokenizer:  corresponding AutoTokenizer
        batch_size: kept small (4) to avoid VRAM OOM with output_hidden_states=True

    Returns:
        np.ndarray of shape (n_layers + 1, hidden_dim)
        Index 0 = embedding layer output; indices 1..N = transformer layers.
        Values are the mean across all input sentences.
    """
    model.eval()
    all_hidden = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        inputs = tokenizer(
            batch, return_tensors='pt', padding=True,
            truncation=True, max_length=MAX_LENGTH
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        hidden_states  = outputs.hidden_states
        attention_mask = inputs['attention_mask']

        for sentence_idx in range(len(batch)):
            mask = attention_mask[sentence_idx].unsqueeze(-1).float()
            n_tokens = mask.sum()
            per_layer = []
            for layer_hs in hidden_states:
                pooled = (layer_hs[sentence_idx] * mask).sum(0) / n_tokens
                per_layer.append(pooled.cpu().float().numpy())  # immediately to CPU
            all_hidden.append(np.stack(per_layer))

        # Free GPU memory after every batch
        del outputs, hidden_states, inputs
        torch.cuda.empty_cache()

    stacked = np.stack(all_hidden)
    return stacked.mean(axis=0)

### 3b. Compute Language Directions

In [ ]:
def compute_language_directions(lang_sentences_dict, english_sentences, model, tokenizer):
    """
    For each target language, compute a direction vector per layer:
        direction[layer] = mean_hidden(lang) - mean_hidden(English)

    Returns:
        dict: lang_name -> np.ndarray of shape (n_layers+1, hidden_dim)
    """
    print('  Computing English mean hidden states...')
    english_mean = get_mean_hidden_states(english_sentences, model, tokenizer)

    directions = {}
    for lang_name, sentences in tqdm(lang_sentences_dict.items(), desc='  Computing directions'):
        lang_mean = get_mean_hidden_states(sentences, model, tokenizer)
        directions[lang_name] = lang_mean - english_mean  # shape: (n_layers+1, hidden_dim)

    return directions

### 3c. Direction Injection Hook

In [ ]:
def make_injection_hook(direction_vector, scale=INJECTION_SCALE):
    """
    Returns a PyTorch forward hook that adds a scaled language direction
    vector to the output of a transformer layer (residual stream injection).

    The hook is registered on a specific transformer block via
    register_forward_hook(). After the forward pass, the hook must be
    removed with handle.remove() to avoid affecting subsequent passes.

    dtype is set to float16 to match the model's computation dtype.
    Using float32 here causes a dtype mismatch RuntimeError.

    Args:
        direction_vector: np.ndarray of shape (hidden_dim,) — the language
                          direction at the layer being injected
        scale:            injection strength (alpha). Set to 20.0; values
                          above ~15 may produce non-linear saturation effects
                          (see Arabic sensitivity > 1.0 in results).

    Returns:
        A hook function compatible with nn.Module.register_forward_hook()
    """
    direction_tensor = torch.tensor(direction_vector, dtype=torch.float16).to(device)  # must match model dtype

    def hook(module, input, output):
        if isinstance(output, tuple):
            hidden = output[0]
            modified = hidden + scale * direction_tensor.unsqueeze(0).unsqueeze(0)
            return (modified,) + output[1:]
        else:
            return output + scale * direction_tensor.unsqueeze(0).unsqueeze(0)

    return hook

### 3d. Sensitivity Measurement (Batched)

Key optimisation: instead of running one sentence at a time through 32 layers,
we batch all injection sentences together. This reduces GPU round-trips from
`n_sentences * n_layers` to just `n_layers`, giving a ~20x speedup.

In [ ]:
def cosine_similarity_np(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)


def get_pooled_final_states(inputs, model):
    """
    Run a batched forward pass and return mean-pooled final hidden states.
    Returns np.ndarray of shape (batch_size, hidden_dim).
    """
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    final_hs = outputs.hidden_states[-1]  # (batch, seq, hidden)
    mask = inputs['attention_mask'].unsqueeze(-1).float()  # (batch, seq, 1)
    pooled = (final_hs * mask).sum(1) / mask.sum(1)  # (batch, hidden)
    return pooled.cpu().float().numpy()


def measure_sensitivity_batched(injection_sentences, direction_vectors, model, tokenizer):
    """
    Batched version: runs all injection sentences together per layer.
    Reduces forward passes from n_sentences*n_layers to n_layers.

    Returns:
        np.ndarray of shape (n_transformer_layers,)
    """
    model.eval()
    transformer_layers = model.model.layers
    n_layers = len(transformer_layers)

    # Tokenise all sentences into one batch
    inputs = tokenizer(
        injection_sentences,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH
    ).to(device)

    # Baseline: one forward pass with no injection — establishes reference representations
    baseline_reps = get_pooled_final_states(inputs, model)  # (n_sents, hidden_dim)

    # Sweep injection layer by layer
    sensitivity_scores = []

    for layer_idx in tqdm(range(n_layers), desc='    Sweeping layers', leave=False):
        # direction index is layer_idx+1 because index 0 in direction_vectors
        # is the embedding layer, not a transformer layer
        direction = direction_vectors[layer_idx + 1]
        hook_fn = make_injection_hook(direction)
        handle = transformer_layers[layer_idx].register_forward_hook(hook_fn)

        injected_reps = get_pooled_final_states(inputs, model)  # (n_sents, hidden)
        handle.remove()

        # Sensitivity = mean(1 - cosine_sim) across sentences.
        # A value of 0 means injection had no effect on final representations.
        # A value > 1 means the representation was pushed past orthogonality
        # (observed for Arabic at layer 5 with alpha=20.0).
        sims = [
            cosine_similarity_np(injected_reps[i], baseline_reps[i])
            for i in range(len(injection_sentences))
        ]
        sensitivity_scores.append(1.0 - np.mean(sims))

    return np.array(sensitivity_scores)

## 4. Run the Analysis

Main loop: for each model variant, load it, compute directions, run the injection sweep, then unload to free VRAM.

**Estimated runtime on T4:** ~1-1.5 hours total.

In [ ]:
# results[variant_name][lang_name] = np.ndarray of shape (n_layers,)
results = defaultdict(dict)

for variant_name, model_id in MODEL_VARIANTS.items():
    print('\n' + '='*60)
    print(f'Model variant: {variant_name} ({model_id})')
    print('='*60)

    print('Loading model and tokenizer...')
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map='auto'
    )
    model.eval()
    n_layers = len(model.model.layers)
    print(f'Model loaded. Transformer layers: {n_layers}')

    print('\nComputing language directions...')
    directions = compute_language_directions(
        lang_direction_sentences,
        english_direction_sentences,
        model,
        tokenizer
    )

    print('\nRunning injection sweep...')
    for lang_name in tqdm(TARGET_LANGUAGES.keys(), desc='Languages'):
        sensitivity = measure_sensitivity_batched(
            lang_injection_sentences[lang_name],
            directions[lang_name],
            model,
            tokenizer
        )
        results[variant_name][lang_name] = sensitivity
        peak = np.argmax(sensitivity)
        score = np.max(sensitivity)
        print(f'  {lang_name}: peak sensitivity at layer {peak} (score={score:.4f})')

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f'\nFinished {variant_name}. VRAM freed.')

print('\nAll variants complete!')

## 5. Save Raw Results

In [ ]:
import json

results_serialisable = {
    variant: {lang: scores.tolist() for lang, scores in langs.items()}
    for variant, langs in results.items()
}

with open('sensitivity_results.json', 'w') as f:
    json.dump(results_serialisable, f, indent=2)

print('Results saved to sensitivity_results.json')

## 6. Visualisation

### 6a. Per-variant: Sensitivity curves coloured by region

In [ ]:
def plot_variant_sensitivity(variant_name, lang_results, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))

    # Group languages by region for consistent colouring
    region_line_styles = ['-', '--', ':']
    region_lang_count = defaultdict(int)

    for lang_name, scores in lang_results.items():
        region = LANGUAGE_REGIONS.get(lang_name, 'Other')
        color = REGION_COLORS.get(region, 'gray')
        linestyle = region_line_styles[region_lang_count[region] % len(region_line_styles)]
        region_lang_count[region] += 1
        n = len(scores)
        ax.plot(range(n), scores, label=lang_name, color=color,
                linestyle=linestyle, linewidth=1.8, alpha=0.9)

    ax.set_title(variant_name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Injection Layer', fontsize=11)
    ax.set_ylabel('Sensitivity (1 - cosine sim)', fontsize=11)
    ax.legend(fontsize=8, loc='upper left', ncol=2)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, n - 1)


fig, axes = plt.subplots(2, 2, figsize=(18, 10))
axes = axes.flatten()

for ax, (variant_name, lang_results) in zip(axes, results.items()):
    plot_variant_sensitivity(variant_name, lang_results, ax=ax)

# Add region colour legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color=c, linewidth=2, label=r)
                   for r, c in REGION_COLORS.items()]
fig.legend(handles=legend_elements, title='Region', loc='lower center',
           ncol=4, fontsize=10, bbox_to_anchor=(0.5, -0.03))

fig.suptitle(
    'Layer Sensitivity Profiling with Direction Injection\nTiny Aya - All Variants',
    fontsize=15, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('sensitivity_by_variant.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sensitivity_by_variant.png')

### 6b. Per-language: Compare sensitivity across all 4 variants

In [ ]:
variant_colors = {'Global': '#1f77b4', 'Water': '#17becf', 'Earth': '#8c564b', 'Fire': '#d62728'}

n_langs = len(TARGET_LANGUAGES)
ncols = 3
nrows = int(np.ceil(n_langs / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
axes = axes.flatten()

for idx, lang_name in enumerate(TARGET_LANGUAGES.keys()):
    ax = axes[idx]
    region = LANGUAGE_REGIONS.get(lang_name, '')
    for variant_name, lang_results in results.items():
        if lang_name in lang_results:
            scores = lang_results[lang_name]
            ax.plot(
                range(len(scores)), scores,
                label=variant_name,
                color=variant_colors.get(variant_name, 'gray'),
                linewidth=2
            )
    ax.set_title(f'{lang_name}  [{region}]', fontsize=11, fontweight='bold')
    ax.set_xlabel('Injection Layer', fontsize=9)
    ax.set_ylabel('Sensitivity', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

for idx in range(n_langs, len(axes)):
    axes[idx].set_visible(False)

fig.suptitle(
    'Per-Language Sensitivity: Global vs Regional Variants',
    fontsize=15, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('sensitivity_by_language.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sensitivity_by_language.png')

### 6c. Heatmap: Peak sensitivity layer per language x variant

In [ ]:
lang_names    = list(TARGET_LANGUAGES.keys())
variant_names = list(MODEL_VARIANTS.keys())

peak_matrix = np.zeros((len(lang_names), len(variant_names)))
for j, variant in enumerate(variant_names):
    for i, lang in enumerate(lang_names):
        if lang in results[variant]:
            scores = results[variant][lang]
            peak_matrix[i, j] = np.argmax(scores) / (len(scores) - 1)

fig, ax = plt.subplots(figsize=(8, 10))
im = ax.imshow(peak_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Normalised peak sensitivity layer (0=early, 1=late)')

ax.set_xticks(range(len(variant_names)))
ax.set_xticklabels(variant_names, fontsize=11)
ax.set_yticks(range(len(lang_names)))

# Add region labels next to language names
ylabels = [f'{lang}  ({LANGUAGE_REGIONS.get(lang, "")})' for lang in lang_names]
ax.set_yticklabels(ylabels, fontsize=10)
ax.set_title('Peak Sensitivity Layer (Normalised by Model Depth)', fontsize=13, fontweight='bold')

for i in range(len(lang_names)):
    for j in range(len(variant_names)):
        ax.text(j, i, f'{peak_matrix[i, j]:.2f}', ha='center', va='center',
                color='black' if peak_matrix[i, j] < 0.6 else 'white', fontsize=9)

plt.tight_layout()
plt.savefig('peak_layer_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: peak_layer_heatmap.png')

## 7. Summary Table

In [ ]:
lang_names    = list(TARGET_LANGUAGES.keys())
variant_names = list(MODEL_VARIANTS.keys())

header = f'  Language         Region        ' + ''.join(f'{v:>16}' for v in variant_names)
print('='*90)
print(header)
print('  (peak layer / max score)')
print('-'*90)

for lang in lang_names:
    region = LANGUAGE_REGIONS.get(lang, '')
    row = f'  {lang:<16}  {region:<12}  '
    for variant in variant_names:
        if lang in results[variant]:
            scores = results[variant][lang]
            peak_layer = np.argmax(scores)
            max_score  = np.max(scores)
            row += f'   L{peak_layer:02d}/{max_score:.3f}   '
        else:
            row += f'      N/A        '
    print(row)

print('='*90)

## 8. Download Results

In [ ]:
from google.colab import files

files.download('sensitivity_results.json')
files.download('sensitivity_by_variant.png')
files.download('sensitivity_by_language.png')
files.download('peak_layer_heatmap.png')